In [1]:
print("Aayush")

Aayush


In [18]:
import os
os.chdir("../")

In [ ]:
# Already in the correct directory from cell 2

In [22]:
%pwd

'c:\\Users\\AAYUSH\\OneDrive\\Desktop\\MedChatbot\\MedChatbot'

In [29]:
os.chdir("../")
os.chdir("../MedChatbot/MedChatbot/data")

In [30]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [31]:
def load_pdf(data):
    loader = DirectoryLoader(data, glob="**/*.pdf", show_progress=True, silent_errors=True, loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [32]:
extracted_docs = load_pdf("../data")

  0%|          | 0/12 [00:00<?, ?it/s]

100%|██████████| 12/12 [08:38<00:00, 43.18s/it]


In [33]:
len(extracted_docs)
    

2648

In [35]:
from typing import List
from langchain_core.documents import Document

def filter_min_docs(docs: List[Document]) -> List[Document]:
    minimal_docs : List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(Document(page_content=doc.page_content, metadata={"source": src}))
    return minimal_docs

In [36]:
minimal_docs = filter_min_docs(extracted_docs)

In [37]:
minimal_docs

[Document(metadata={'source': '..\\data\\Adenocarcinoma.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\Adenocarcinoma.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\Adenocarcinoma.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\Adenocarcinoma.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\Adenocarcinoma.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\Adenocarcinoma.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\Adenocarcinoma.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\Adenocarcinoma.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\Adenocarcinoma.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\Adenocarcinoma.pdf'}, page_content=''),
 Document(metadata={'source': '..\\data\\Fibrosis.pdf'}, page_content='AMERICAN THORACIC SOCIETY\nDOCUMENTS\nIdiopathic Pulmonary Fibrosis (an Update) and Progressive\nPulmonary Fibrosis in 

In [38]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100,length_function=len,separators=["\n\n", "\n", ".", " ", ""])
    text_chunk = text_splitter.split_documents(minimal_docs)
    return text_chunk


In [39]:
text_chunk = text_split(minimal_docs)

In [40]:
print(len(text_chunk))

10116


In [41]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "pritamdeka/S-PubMedBert-MS-MARCO"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"✅ Using device: {device}")
    
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={
            "device": device        
        },
        encode_kwargs={
            "normalize_embeddings": True
        }
    )
    print(f"✅ Embeddings loaded: {model_name}")
    return embeddings

embedding = download_embeddings()

✅ Using device: cpu
✅ Embeddings loaded: pritamdeka/S-PubMedBert-MS-MARCO


In [42]:
embedding

HuggingFaceEmbeddings(model_name='pritamdeka/S-PubMedBert-MS-MARCO', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [43]:
vector = embedding.embed_query("What is the role of ACE2 in COVID-19?")
vector

[-0.01070345751941204,
 -0.03855503350496292,
 -0.055344197899103165,
 -0.048208680003881454,
 -0.03312290459871292,
 -0.012821362353861332,
 -0.05287664011120796,
 0.028829555958509445,
 0.020193669945001602,
 0.012240733951330185,
 0.010274793021380901,
 0.01837282069027424,
 0.008808702230453491,
 -0.017190316691994667,
 -0.043902814388275146,
 0.007927841506898403,
 0.0026325201615691185,
 0.026606211438775063,
 -0.020324580371379852,
 0.05956804007291794,
 0.004983746446669102,
 0.0017780654598027468,
 -0.0042199925519526005,
 -0.00920841284096241,
 -0.015163141302764416,
 0.04353633522987366,
 0.0030161950271576643,
 0.0141709940508008,
 0.008889374323189259,
 -0.03776676207780838,
 0.03644800931215286,
 0.011873429641127586,
 -0.0035269176587462425,
 0.020040152594447136,
 0.03624187409877777,
 -0.01103498786687851,
 -0.01574379950761795,
 -0.022222032770514488,
 0.05066428333520889,
 0.012518437579274178,
 -0.0029293056577444077,
 0.013769284822046757,
 0.02989407256245613,
 -0

In [44]:
print(len(vector))

768


In [45]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [46]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [47]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)

In [48]:
pc

In [49]:
from pinecone import ServerlessSpec
index_name = "medchatbot"
if not pc.has_index(index_name):
    pc.create_index(name=index_name, dimension=768, metric="cosine", spec=ServerlessSpec(cloud="aws",region="us-east-1"))
index = pc.Index(index_name)

In [52]:
from langchain_pinecone import PineconeVectorStore

# Upload documents in batches to avoid size limit
batch_size = 50
docsearch = None

for i in range(0, len(text_chunk), batch_size):
    batch = text_chunk[i:i + batch_size]
    print(f"Uploading batch {i//batch_size + 1}/{(len(text_chunk)-1)//batch_size + 1}...")
    
    if docsearch is None:
        # Create the vectorstore with the first batch
        docsearch = PineconeVectorStore.from_documents(
            documents=batch, 
            embedding=embedding, 
            index_name=index_name
        )
    else:
        # Add subsequent batches
        docsearch.add_documents(batch)

print("✅ All documents uploaded to Pinecone!")

Uploading batch 1/203...
Uploading batch 2/203...
Uploading batch 3/203...
Uploading batch 4/203...
Uploading batch 5/203...
Uploading batch 6/203...
Uploading batch 7/203...
Uploading batch 8/203...
Uploading batch 9/203...
Uploading batch 10/203...
Uploading batch 11/203...
Uploading batch 12/203...
Uploading batch 13/203...
Uploading batch 14/203...
Uploading batch 15/203...
Uploading batch 16/203...
Uploading batch 17/203...
Uploading batch 18/203...
Uploading batch 19/203...
Uploading batch 20/203...
Uploading batch 21/203...
Uploading batch 22/203...
Uploading batch 23/203...
Uploading batch 24/203...
Uploading batch 25/203...
Uploading batch 26/203...
Uploading batch 27/203...
Uploading batch 28/203...
Uploading batch 29/203...
Uploading batch 30/203...
Uploading batch 31/203...
Uploading batch 32/203...
Uploading batch 33/203...
Uploading batch 34/203...
Uploading batch 35/203...
Uploading batch 36/203...
Uploading batch 37/203...
Uploading batch 38/203...
Uploading batch 39/20

In [53]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(embedding=embedding, index_name=index_name)


In [54]:
# IF want to add new documents in the future, use this code.
# docsearch.add_documents(documents=[])

In [55]:
retriever = docsearch.as_retriever(search_type = "similarity",search_kwargs={"k": 3})
retrieve_docs = retriever.invoke("What is the role of ACE2 in COVID-19?")


In [56]:
retrieve_docs

[Document(id='b32fe0dc-fba6-4eed-a37a-96d0f54d4c70', metadata={'source': '..\\data\\GOLD-2024_v1.2-11Jan24_WMV.pdf'}, page_content='recommends the Tdap vaccination (also called dTaP/ dTPa) to protect against pertussis, in addition to tetanus and \ndiphtheria, in those who were not vaccinated in adolescence, together with routine shingles vaccination.(645) \n \nPeople with COPD should have the COVID-19 vaccinations in line with national recommendations. (645,668) COVID-19 \nvaccines are highly effective against SARS-CoV-2 infection requiring hospitalization, ICU admission, or an ED or urgent \ncare clinic visit, including those with chronic respiratory disease.(190,669) \nEducation and self-management \nEducation, self-management and integrative care  \nPatient “education” often takes the form of providers giving information and advice, and assumes that knowledge will \nlead to behavior change. Although enhancing patient knowledge is an important step towards behavior change, \ndidactic

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

# ✅ Free LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=groq_api_key,
    temperature=0.4,
    max_tokens=1024
)

print("✅ LLM loaded — Groq Llama3 70B (free)")

# Quick test
response = llm.invoke("What is COPD in one sentence?")
print(response.content)

✅ LLM loaded — Groq Llama3 70B (free)
COPD, or Chronic Obstructive Pulmonary Disease, is a progressive lung disease that makes it difficult to breathe and can cause symptoms such as wheezing, coughing, and shortness of breath, often resulting from long-term exposure to lung irritants like cigarette smoke.


In [66]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [67]:
system_prompt = (
    "You are a helpful and precise assistant for answering questions about medical research papers. "
    "Use the following retrieved document excerpts to provide a concise and accurate answer to the question. "
    "If you don't know the answer, say you don't know. Always use all available data to answer the question."
    "Use three sentences maximum to answer the question. Always use all available data to answer the question."
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human","{input}")
])

In [68]:
llm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024201D4F1D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000242025F4050>, model_name='llama-3.3-70b-versatile', temperature=0.4, model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=1024)

In [70]:
question_ans_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)
rag_chain = create_retrieval_chain(retriever, question_ans_chain)

In [76]:
response = rag_chain.invoke({"input": "What is Tubercolosis and what are its symptoms?"})
print(response["answer"])

Tuberculosis (TB) is a disease in humans caused by the M. tuberculosis complex, which comprises eight distinct but closely related organisms. The symptoms of Pulmonary Tuberculosis (PTB) are protean and often associated with nonspecific constitutional and specific respiratory signs and symptoms, including tiredness, malaise, weight loss, fever, night sweats, anorexia, and headache. Common symptoms also include cough, which is the most predominant symptom, and may be accompanied by the production of mucoid or mucopurulent sputum, and occasionally hemoptysis (coughing up blood).
